In [2]:
import os, math, time, random, warnings, gc
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
from albumentations import (
    HorizontalFlip, RandomBrightnessContrast, GaussianBlur,
    ShiftScaleRotate, Resize, Normalize, Compose
)
from albumentations.pytorch import ToTensorV2

from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [3]:
VOC_ROOT   = Path("/kaggle/input/pascal-voc-2012-dataset/VOC2012_train_val/VOC2012_train_val")
IMG_ROOT   = VOC_ROOT / "JPEGImages"
GT_ROOT    = VOC_ROOT / "SegmentationClass"
SPLIT_ROOT = VOC_ROOT / "ImageSets" / "Segmentation"

assert IMG_ROOT.exists() and GT_ROOT.exists() and SPLIT_ROOT.exists(), "VOC folders missing."

def read_ids(txt_path: Path):
    with open(txt_path) as f:
        return [x.strip() for x in f if x.strip()]

train_ids = read_ids(SPLIT_ROOT / "train.txt")
val_ids   = read_ids(SPLIT_ROOT / "val.txt")
print(f"IDs → train={len(train_ids)}, val={len(val_ids)} (expect ~1464 / 1449)")

EXP_NAME    = "E_dilate5"
BASE_OUT    = Path("/kaggle/working/outputs") / EXP_NAME
SEEDS_DIR   = Path("/kaggle/working/outputs/seed_overlays")       # viz only (optional)
PSEUDO_SRC  = Path("/kaggle/working/outputs/pseudo_masks/gradcam") # Grad-CAM seeds live here
PSEUDO_DIR  = Path("/kaggle/working/outputs/pseudo_masks/gradcam_dilate5")  # this exp's masks
RESULTS_DIR = BASE_OUT / "results"
SAMPLE_DIR  = BASE_OUT / "samples"
CKPT_PATH   = BASE_OUT / "deeplab_binary_best.pth"

for d in [BASE_OUT, RESULTS_DIR, SAMPLE_DIR, SEEDS_DIR, PSEUDO_SRC, PSEUDO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", BASE_OUT)

IDs → train=1464, val=1449 (expect ~1464 / 1449)
Saving outputs to: /kaggle/working/outputs/E_dilate5


In [4]:
import torchvision
from torchvision.models import resnet50, ResNet50_Weights

class CAMHelper:
    def __init__(self):
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(DEVICE).eval()
        self.model = m
        self.feats, self.grads = [], []
        def f_hook(_, __, out): self.feats.append(out.detach())
        def b_hook(_, grad_in, grad_out): self.grads.append(grad_out[0].detach())
        self.handles = [
            m.layer4[-1].conv3.register_forward_hook(f_hook),
            m.layer4[-1].conv3.register_full_backward_hook(b_hook),
        ]
        self.pre = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=[0.485,0.456,0.406],
                                             std=[0.229,0.224,0.225])
        ])
    def __call__(self, pil_img: Image.Image) -> np.ndarray:
        self.feats.clear(); self.grads.clear()
        x = self.pre(pil_img).unsqueeze(0).to(DEVICE)
        logits = self.model(x)
        cls = logits.argmax(dim=1)
        logits[0, cls].backward()
        A = self.feats[-1][0]; G = self.grads[-1][0]
        w = G.mean(dim=(1,2))
        cam = torch.relu((w[:,None,None]*A).sum(0))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        return cam.detach().cpu().numpy()
    def close(self):
        for h in self.handles: h.remove()

def build_gradcam_masks(ids, th=0.30):
    cam_helper = CAMHelper()
    wrote = 0
    for img_id in tqdm(ids, desc="Grad-CAM seeds", unit="img"):
        ip = IMG_ROOT / f"{img_id}.jpg"
        if not ip.exists(): ip = IMG_ROOT / f"{img_id}.jpeg"
        if not ip.exists(): continue
        img = Image.open(ip).convert("RGB")
        W,H = img.size
        cmap = cam_helper(img)                       # [h,w] (feature resolution)
        cam_up = cv2.resize(cmap, (W,H), interpolation=cv2.INTER_LINEAR)
        mask = (cam_up >= th).astype(np.uint8) * 255
        Image.fromarray(mask).save(PSEUDO_SRC / f"{img_id}.png")
        wrote += 1
    cam_helper.close()
    print(f"Grad-CAM masks saved → {PSEUDO_SRC}  (count={wrote})")


existing = list(PSEUDO_SRC.glob("*.png"))
print("Existing Grad-CAM seeds:", len(existing))
if len(existing) < 100:     
    build_gradcam_masks(train_ids, th=0.30)
else:
    print("Grad-CAM seeds already present — skipping rebuild.")

Existing Grad-CAM seeds: 0


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 165MB/s] 


Grad-CAM seeds:   0%|          | 0/1464 [00:00<?, ?img/s]

Grad-CAM masks saved → /kaggle/working/outputs/pseudo_masks/gradcam  (count=1464)


In [5]:
IGNORE_IDX = 255
IMG_SIZE   = 256
BATCH_TRAIN = 8
BATCH_VAL   = 8

class VOCPseudoBinary(Dataset):
    """
    x: FloatTensor [3,H,W]
    g: LongTensor  [H,W] in {0,1}      (pseudo binarized)
    q: LongTensor  [H,W] in {0,1,255}  (GT-for-eval, 255=ignore)
    id: str
    """
    def __init__(self, ids, img_root, pseudo_root, gt_root, train=True, size=256):
        self.ids = ids
        self.img_root    = Path(img_root)
        self.pseudo_root = Path(pseudo_root)
        self.gt_root     = Path(gt_root)
        self.train       = bool(train)
        self.size        = int(size)

        aug_train = Compose([
            HorizontalFlip(p=0.5),
            RandomBrightnessContrast(p=0.2),
            GaussianBlur(blur_limit=(3,5), p=0.15),
            ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15,
                             border_mode=cv2.BORDER_CONSTANT, p=0.5),
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        aug_val = Compose([
            Resize(self.size, self.size),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
        self.tr = aug_train if self.train else aug_val

    def __len__(self): return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]

        ip = self.img_root / f"{img_id}.jpg"
        if not ip.exists(): ip = self.img_root / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB")); H,W = img.shape[:2]

        pp = self.pseudo_root / f"{img_id}.png"
        if pp.exists(): pm = np.array(Image.open(pp))
        else:           pm = np.zeros((H,W), np.uint8)

        gp = self.gt_root / f"{img_id}.png"
        if gp.exists(): gt = np.array(Image.open(gp))
        else:           gt = np.full((H,W), IGNORE_IDX, np.uint8)

   
        if pm.shape != (H,W): pm = cv2.resize(pm, (W,H), interpolation=cv2.INTER_NEAREST)
        if gt.shape != (H,W): gt = cv2.resize(gt, (W,H), interpolation=cv2.INTER_NEAREST)

        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))

        out = self.tr(image=img, mask=pm, masks=[gbin])
        x   = out["image"].float()
        pm2 = out["mask"]
        gt2 = out["masks"][0]

        pm2 = torch.as_tensor(pm2).squeeze()
        if pm2.dtype.is_floating_point:
            g = (pm2 > 0.5).long()
        else:
            g = (pm2 > 0).long()

        if isinstance(gt2, torch.Tensor):
            q = gt2.squeeze().long()
        else:
            q = torch.from_numpy(gt2.astype(np.uint8)).long()

        return x, g, q, img_id

In [6]:
def build_dilate(src_dir: Path, dst_dir: Path, ids, kernel_radius_px=5):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_radius_px*2+1,)*2)
    wrote, missing = 0, 0
    for img_id in tqdm(ids, desc=f"Dilate r={kernel_radius_px}", unit="img"):
        src = src_dir / f"{img_id}.png"
        if not src.exists():
            missing += 1
            continue
        arr = np.array(Image.open(src))
        if arr.ndim == 3: arr = arr[...,0]
        fg  = (arr > 0).astype(np.uint8)
        dil = cv2.dilate(fg, kernel, iterations=1)
        out = (dil * 255).astype(np.uint8)
        Image.fromarray(out).save(dst_dir / f"{img_id}.png")
        wrote += 1
    print(f"dilate{kernel_radius_px} → wrote {wrote} masks to {dst_dir}")
    if missing: print(f"WARNING: {missing} Grad-CAM seeds missing in {src_dir}")


paths = list(PSEUDO_SRC.glob("*.png"))
nonempty = sum(np.array(Image.open(p)).max()>0 for p in paths[:100])
assert len(paths)>100 and nonempty>10, "No Grad-CAM masks found. Build seeds first."

build_dilate(PSEUDO_SRC, PSEUDO_DIR, train_ids, kernel_radius_px=5)

def make_loaders(num_workers=2, pin=True):
    train_ds = VOCPseudoBinary(train_ids, IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=True,  size=IMG_SIZE)
    val_ds   = VOCPseudoBinary(val_ids,   IMG_ROOT, PSEUDO_DIR, GT_ROOT, train=False, size=IMG_SIZE)
    train_dl = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,
                          num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    val_dl   = DataLoader(val_ds, batch_size=BATCH_VAL,   shuffle=False,
                          num_workers=num_workers, pin_memory=pin, persistent_workers=False)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()

fg_pix, tot_pix = 0, 0
for step,(x,g,_,_) in enumerate(train_dl):
    fg_pix += (g>0).sum().item()
    tot_pix += g.numel()
    if step==9: break
frac = fg_pix/max(1,tot_pix)
print(f"[probe] pseudo>0 pixels: {fg_pix} ({frac:.2%})")

Dilate r=5:   0%|          | 0/1464 [00:00<?, ?img/s]

dilate5 → wrote 1464 masks to /kaggle/working/outputs/pseudo_masks/gradcam_dilate5
[probe] pseudo>0 pixels: 1667549 (31.81%)


In [8]:
from torchvision.models.segmentation import deeplabv3_resnet50

def build_deeplab_binary(num_classes=2):
    m = deeplabv3_resnet50(weights_backbone=None, num_classes=num_classes)
    return m.to(DEVICE)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, target01):
        # logits: [B,2,H,W]; target01: [B,H,W] in {0,1}
        probs = torch.softmax(logits, dim=1)[:,1]      # foreground prob
        target = target01.float()
        num = 2*(probs*target).sum(dim=(1,2)) + self.smooth
        den = (probs.pow(2)+target.pow(2)).sum(dim=(1,2)) + self.smooth
        return 1 - (num/den).mean()

@torch.no_grad()
def iou_from_logits(logits, target01, ignore=None):
    pred = logits.argmax(1)
    if ignore is not None:
        mask = target01!=ignore
        pred = pred[mask]; tgt = target01[mask]
    else:
        tgt = target01
    inter = ((pred==1) & (tgt==1)).sum().float()
    union = ((pred==1) | (tgt==1)).sum().float().clamp_min(1)
    bg_inter = ((pred==0) & (tgt==0)).sum().float()
    bg_union = ((pred==0) | (tgt==0)).sum().float().clamp_min(1)
    return (bg_inter/bg_union).item(), (inter/union).item()

In [9]:
LR = 1e-3
EPOCHS = 15
PATIENCE = 5

model = build_deeplab_binary(num_classes=2)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = DiceLoss()

def evaluate(val_loader):
    model.eval()
    bg_iou, fg_iou, n = 0.0, 0.0, 0
    with torch.no_grad():
        for x,_,q,_ in val_loader:
            x = x.to(DEVICE)
            q = q.to(DEVICE)
            logits = model(x)['out']
            bgi, fgi = iou_from_logits(logits, torch.where(q==IGNORE_IDX, 0, q), ignore=None)
            bg_iou += bgi; fg_iou += fgi; n += 1
    return {"IoU_bg": bg_iou/max(1,n), "IoU_fg": fg_iou/max(1,n), "mIoU": (bg_iou+fg_iou)/max(1,2*n)}

best_miou, wait = -1, 0
for ep in range(1, EPOCHS+1):
    model.train()
    pbar = tqdm(train_dl, desc=f"Epoch {ep:02d}/{EPOCHS}", leave=False)
    total = 0.0
    for x,g,_,_ in pbar:
        x = x.to(DEVICE)
        g = g.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        out = model(x)['out']                         # [B,2,H,W]
        loss = criterion(out, g)
        loss.backward()
        opt.step()
        total += loss.item()*x.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg_loss = total/len(train_dl.dataset)

    mets = evaluate(val_dl)
    print(f"Epoch {ep:02d} | loss={avg_loss:.4f} | mIoU={mets['mIoU']:.3f} (bg={mets['IoU_bg']:.3f}, fg={mets['IoU_fg']:.3f})")
    if mets["mIoU"] > best_miou + 1e-4:
        best_miou = mets["mIoU"]; wait = 0
        torch.save(model.state_dict(), CKPT_PATH)
        print("  New best; checkpoint saved ->", CKPT_PATH.name)
    else:
        wait += 1
        if wait > PATIENCE:
            print("Early stop ")
            break

print("Best mIoU:", round(best_miou,3))

Epoch 01/15:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 01 | loss=0.3708 | mIoU=0.478 (bg=0.597, fg=0.358)
  New best; checkpoint saved -> deeplab_binary_best.pth


Epoch 02/15:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 02 | loss=0.3536 | mIoU=0.455 (bg=0.564, fg=0.346)


Epoch 03/15:   0%|          | 0/183 [00:00<?, ?it/s]

...

Epoch 13 | loss=0.3273 | mIoU=0.467 (bg=0.588, fg=0.346)


Epoch 14/15:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 14 | loss=0.3267 | mIoU=0.491 (bg=0.620, fg=0.362)


Epoch 15/15:   0%|          | 0/183 [00:00<?, ?it/s]

Epoch 15 | loss=0.3258 | mIoU=0.477 (bg=0.600, fg=0.354)
Best mIoU: 0.493


In [10]:
import pandas as pd

def perturbations(img):
    out = {}
    out["clean"] = img
    out["blur"]  = cv2.GaussianBlur(img, (5,5), 1.0)
    out["brightness"] = np.clip(img.astype(np.float32)*1.25, 0, 255).astype(np.uint8)
    out["gauss"] = np.clip(img.astype(np.float32) + np.random.normal(0, 10, img.shape), 0, 255).astype(np.uint8)
    out["hflip"] = img[:, ::-1, :]
    out["rotation"] = cv2.warpAffine(
        img, cv2.getRotationMatrix2D((img.shape[1]/2, img.shape[0]/2), 15, 1.0),
        (img.shape[1], img.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101
    )
    return out

@torch.no_grad()
def eval_under_perturbations(val_ids, max_items=None, sample_ratio=1.0):
    model.eval()
    rows = []
    ids = val_ids if max_items is None else val_ids[:max_items]
    ids = ids[:int(len(ids)*sample_ratio)]
    pbar = tqdm(ids, desc="Robustness", unit="img")
    for img_id in pbar:
        ip = IMG_ROOT / f"{img_id}.jpg"
        if not ip.exists(): ip = IMG_ROOT / f"{img_id}.jpeg"
        img = np.array(Image.open(ip).convert("RGB"))
        H,W = img.shape[:2]
        gp = GT_ROOT / f"{img_id}.png"
        gt = np.array(Image.open(gp)) if gp.exists() else np.full((H,W), IGNORE_IDX, np.uint8)
        gbin = np.where(gt==IGNORE_IDX, IGNORE_IDX, (gt!=0).astype(np.uint8))

        for name,im in perturbations(img).items():
       
            x = Compose([
                Resize(IMG_SIZE, IMG_SIZE),
                Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                ToTensorV2()
            ])(image=im)["image"].unsqueeze(0).to(DEVICE)

            logits = model(x)['out']
        
            gtr = cv2.resize(gbin, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            bgi,fgi = iou_from_logits(logits, torch.from_numpy(gtr).long().to(DEVICE), ignore=None)
            rows.append((name, bgi, fgi, 0.5*(bgi+fgi)))
    df = pd.DataFrame(rows, columns=["perturb","IoU_bg","IoU_fg","mIoU"])
    df = df.groupby("perturb", as_index=False).mean().sort_values("perturb").reset_index(drop=True)
    out_csv = RESULTS_DIR / "voc_val_robustness.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")
    return df

df_rob = eval_under_perturbations(val_ids, max_items=None)
df_rob

Robustness:   0%|          | 0/1449 [00:00<?, ?img/s]

Saved: /kaggle/working/outputs/E_dilate5/results/voc_val_robustness.csv


,perturb,IoU_bg,IoU_fg,mIoU
0,blur,0.602085,0.337806,0.469946
1,brightness,0.588367,0.336846,0.462606
2,clean,0.597577,0.336934,0.467256
3,gauss,0.597099,0.337052,0.467075
4,hflip,0.570848,0.311234,0.441041
5,rotation,0.588958,0.328179,0.458568
